# NB05：Query 轉換技術 — 讓檢索更準確的四大策略

**目標：** 理解為何查詢品質是 RAG 系統的關鍵瓶頸，並掌握四種主流的 Query Transformation 技術。

**學習成果：**
- 理解「垃圾進、垃圾出」問題對 RAG 的影響
- 掌握 HyDE（假設性文件嵌入）的原理與實作
- 掌握 Multi-Query（多查詢）技術提升召回率
- 掌握 Step-Back Prompting（後退提問）抽象化策略
- 掌握 Query Decomposition（查詢分解）處理複雜多跳問題
- 能根據查詢類型選擇最佳策略

## 1. 核心問題：為什麼好的 RAG 也會失敗？

### 「垃圾進、垃圾出」（Garbage In, Garbage Out）

即使你有完美的向量資料庫、精心調整的 chunk size、最先進的 reranker，如果**使用者輸入的查詢很糟糕**，RAG 系統還是會給出錯誤答案。

```
❌ 問題鏈：
糟糕的查詢
    ↓
錯誤的向量搜索方向
    ↓
召回到不相關的文件
    ↓
LLM 根據錯誤 context 生成答案
    ↓
錯誤的最終回答（即使 LLM 很強也無能為力）
```

### 查詢失敗的三大原因

| 問題類型 | 範例 | 失敗原因 |
|----------|------|----------|
| **模糊/開放查詢** | 「解釋一下 BERT」 | 查詢向量太泛，無法聚焦相關文件 |
| **措辭不匹配** | 「Bidirectional Encoder 的學習」vs「BERT 的預訓練」| 不同措辭但語意相同，向量距離卻可能較遠 |
| **複雜多跳問題** | 「BERT 和 GPT-4 的多語言能力誰更強？」 | 需要先各自了解再比較，單一查詢難以捕捉 |

## 2. 四大 Query 轉換策略概覽

| 策略 | 核心思想 | 最適用場景 | 代價 |
|------|----------|------------|------|
| **HyDE** | 先生成假設答案，用答案的向量去搜索 | 模糊、開放性問題 | +1 次 LLM 呼叫 |
| **Multi-Query** | 將同一問題改寫 N 種方式，取聯集 | 措辭敏感、召回率要求高 | +N 次向量搜索 |
| **Step-Back** | 先問更通用的大問題，取得背景知識 | 需要背景知識才能回答的具體問題 | +1 次 LLM 呼叫 |
| **Query Decomposition** | 拆成子問題依序回答，用前答輔助後問 | 複雜多跳推理問題 | +N 次 LLM 呼叫 |

```
四大策略的適用邏輯：

查詢很模糊、不具體？
  → 用 HyDE：讓 LLM 想像「答案長什麼樣」

查詢措辭可能和文件不匹配？
  → 用 Multi-Query：多種問法都試試，取聯集

查詢需要先理解背景才能回答？
  → 用 Step-Back：先問大問題、取得背景知識

查詢包含多個子問題或需要多步推理？
  → 用 Query Decomposition：拆解後逐一攻破
```

## 3. 環境設定

In [ ]:
import os
import json
import time
import random
from dotenv import load_dotenv
from openai import OpenAI
import chromadb

# 載入 .env 中的 API Key
load_dotenv()

# 初始化 OpenAI client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# 初始化 ChromaDB（in-memory）
chroma_client = chromadb.Client()

print("✅ 環境設定完成")
print(f"OpenAI API Key 存在: {bool(os.getenv('OPENAI_API_KEY'))}")

## 4. 建立示範知識庫

我們使用一批關於 NLP 模型、向量資料庫技術的文件建立知識庫，作為後續四種策略的示範基礎。

In [ ]:
# 示範知識庫文件
documents = [
    {"id": "d01", "content": "BERT 使用遮罩語言模型（MLM）和下一句預測（NSP）兩種預訓練任務。MLM 隨機遮住 15% 的詞彙，NSP 讓模型判斷兩句是否相鄰。BERT 採用雙向 Transformer，能同時考慮前後文。"},
    {"id": "d02", "content": "GPT-4 是 OpenAI 2023 年 3 月發布的多模態模型。在律師資格考試中達到前 10% 成績，在 SAT 數學達到 700 分。GPT-4 Turbo 支援 128K token 上下文窗口。"},
    {"id": "d03", "content": "Transformer 的自注意力機制讓模型動態關注序列不同位置。多頭注意力（Multi-Head Attention）允許模型同時從不同角度理解序列關係。位置編碼解決了 Transformer 無法感知順序的問題。"},
    {"id": "d04", "content": "量子電腦利用量子位元（qubit）進行計算，可同時處於多種狀態（疊加態）。Shor's algorithm 在量子電腦上能多項式時間內分解大數，對 RSA 加密構成威脅。目前 IBM、Google 在量子計算領域領先。"},
    {"id": "d05", "content": "大型語言模型的多語言能力差異顯著。mBERT 支援 104 種語言，XLM-R 在跨語言任務上表現更好。GPT-4 在英文上最強，繁體中文表現中等。Claude 在繁體中文理解上有特別優化。"},
    {"id": "d06", "content": "RAG 系統的延遲主要來源：向量搜索（30-100ms）、Reranking（100-500ms）、LLM 生成（500-3000ms）。優化策略包括：語意快取、非同步檢索、更小的 LLM 模型、減少 context 長度。"},
    {"id": "d07", "content": "向量資料庫的 HNSW 索引使用分層圖結構進行近似最近鄰搜索。搜索時從頂層稀疏圖開始，逐層向下找到最近鄰。HNSW 建立索引慢但搜索快，適合讀多寫少的場景。"},
    {"id": "d08", "content": "Embedding 模型的 Matryoshka Representation Learning（MRL）技術允許將高維向量截斷為低維仍保持相對品質。例如 text-embedding-3-large 的 3072 維可截斷為 256 維使用，大幅降低儲存和計算成本。"},
]

print(f"知識庫共 {len(documents)} 份文件：")
for doc in documents:
    print(f"  [{doc['id']}] {doc['content'][:50]}...")

In [ ]:
def get_embedding(text: str) -> list[float]:
    """使用 OpenAI text-embedding-3-small 生成向量"""
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding

# 建立 ChromaDB collection
try:
    chroma_client.delete_collection("query_transform_demo")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="query_transform_demo",
    metadata={"hnsw:space": "cosine"}
)

# 索引所有文件
print("建立向量索引...")
for doc in documents:
    embedding = get_embedding(doc["content"])
    collection.add(
        ids=[doc["id"]],
        embeddings=[embedding],
        documents=[doc["content"]]
    )
    print(f"  ✅ [{doc['id']}] 已索引")

print(f"\n向量資料庫共 {collection.count()} 份文件")

In [ ]:
def standard_retrieve(query: str, n_results: int = 3) -> list[dict]:
    """標準向量搜索（作為 baseline 對比用）"""
    query_emb = get_embedding(query)
    results = collection.query(
        query_embeddings=[query_emb],
        n_results=n_results,
        include=["documents", "distances"]
    )
    return [
        {"id": id_, "content": doc, "score": round(1 - dist, 4)}
        for id_, doc, dist in zip(
            results["ids"][0],
            results["documents"][0],
            results["distances"][0]
        )
    ]

# 測試 baseline
test = standard_retrieve("量子電腦如何影響加密？")
print("Baseline 搜索結果（量子電腦如何影響加密？）：")
for r in test:
    print(f"  [{r['id']}] 分數={r['score']} | {r['content'][:60]}...")

---
## 策略一：HyDE（Hypothetical Document Embeddings，假設性文件嵌入）

### 核心概念

**問題所在：** 查詢（Query）和文件（Document）在向量空間中天然存在差距。
- 查詢通常很短：「量子電腦如何影響加密？」（8 個字）
- 文件通常較長、結構完整，包含術語和解釋
- 兩者即使語意相關，向量距離也可能較遠

**HyDE 的解法：** 不直接嵌入查詢，而是讓 LLM 先「假裝回答」，生成一個假設性文件，再嵌入這個假設文件去搜索。

```
傳統方式：
  查詢 → [embed] → 查詢向量 → 搜索文件 ← 問題：查詢向量和文件向量風格不同

HyDE 方式：
  查詢 → [LLM 生成假設答案] → 假設文件
         → [embed] → 假設文件向量
         → 搜索真實文件 ← 優勢：假設文件和真實文件風格相似！
```

### 直覺解釋

想像你在圖書館找書，與其拿著問題「量子電腦如何影響加密？」去比對書籍，不如先自己寫一段關於這個主題的草稿，再拿草稿去找和它最像的書——效果往往更好！

In [ ]:
def generate_hypothetical_document(query: str) -> str:
    """讓 LLM 生成一個假設性答案文件，用於 HyDE 搜索"""
    prompt = f"""請針對以下問題，生成一段簡短的假設性答案（約 100-150 字）。
這個答案會用來做語意搜索，不需要 100% 正確，但要使用相關術語和概念。
只輸出答案段落，不要有前綴或解釋。

問題：{query}"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7  # 稍高的溫度，讓假設文件更多樣
    )
    return response.choices[0].message.content.strip()


def hyde_retrieve(query: str, n_results: int = 3) -> dict:
    """HyDE 檢索：先生成假設文件，再用假設文件向量搜索"""
    # 步驟 1：生成假設性答案文件
    hypo_doc = generate_hypothetical_document(query)

    # 步驟 2：嵌入假設文件（而非原始查詢）
    hypo_embedding = get_embedding(hypo_doc)

    # 步驟 3：用假設文件向量搜索真實文件
    results = collection.query(
        query_embeddings=[hypo_embedding],
        n_results=n_results,
        include=["documents", "distances"]
    )

    retrieved = [
        {"id": id_, "content": doc, "score": round(1 - dist, 4)}
        for id_, doc, dist in zip(
            results["ids"][0],
            results["documents"][0],
            results["distances"][0]
        )
    ]
    return {"hypothetical_doc": hypo_doc, "results": retrieved}


print("HyDE 函式已定義 ✅")

In [ ]:
# 示範：模糊查詢的 HyDE 效果
vague_query = "量子電腦如何影響加密？"

print(f"查詢：{vague_query}")
print("=" * 60)

# --- 標準搜索 ---
standard_results = standard_retrieve(vague_query)
print("\n❶ 標準向量搜索結果：")
for r in standard_results:
    print(f"   [{r['id']}] 分數={r['score']} | {r['content'][:55]}...")

# --- HyDE 搜索 ---
hyde_result = hyde_retrieve(vague_query)

print(f"\n❷ HyDE 生成的假設文件：")
print(f"   {hyde_result['hypothetical_doc'][:200]}...")

print(f"\n❸ HyDE 搜索結果：")
for r in hyde_result["results"]:
    print(f"   [{r['id']}] 分數={r['score']} | {r['content'][:55]}...")

# --- 比較 ---
std_ids = {r["id"] for r in standard_results}
hyde_ids = {r["id"] for r in hyde_result["results"]}
print(f"\n📊 比較：標準={std_ids}，HyDE={hyde_ids}")
if hyde_ids != std_ids:
    print(f"   HyDE 找到了不同的文件！差異：{hyde_ids.symmetric_difference(std_ids)}")
else:
    print("   兩者找到相同文件（查詢本身已很清晰）")

In [ ]:
# 進一步對比：HyDE 在精確查詢上的表現（預期差異較小）
precise_query = "BERT 的 MLM 預訓練任務"

print(f"精確查詢：{precise_query}")
print("-" * 50)

std_precise = standard_retrieve(precise_query, n_results=3)
hyde_precise = hyde_retrieve(precise_query, n_results=3)

print("標準搜索 Top-1：", std_precise[0]["id"], "|", std_precise[0]["score"])
print("HyDE 搜索 Top-1 ：", hyde_precise["results"][0]["id"], "|", hyde_precise["results"][0]["score"])
print()
print("結論：對於已經很精確的查詢，HyDE 和標準搜索差異不大，")
print("      HyDE 的額外 LLM 呼叫成本並不值得。")

### HyDE 重點總結

| 面向 | 說明 |
|------|------|
| **優勢** | 模糊問題效果顯著提升；假設文件和真實文件風格一致 |
| **劣勢** | 需要額外一次 LLM 呼叫；若假設文件生成方向錯誤，結果更差 |
| **最適場景** | 開放性問題（「什麼是...」「解釋...」）、使用者描述不清楚的查詢 |
| **不適場景** | 精確查詢（有完整術語）、對延遲極度敏感的系統 |

> **論文來源：** Gao et al., 2022, "Precise Zero-Shot Dense Retrieval without Relevance Labels"

---
## 策略二：Multi-Query Retrieval（多查詢檢索）

### 核心概念

**問題所在：** 向量搜索對查詢的**措辭非常敏感**。同一個問題用不同方式表達，可能找到完全不同的文件。

```
查詢 A：「BERT 的訓練方式」
查詢 B：「Bidirectional Encoder 的預訓練任務」  ← 語意相同，措辭不同
查詢 C：「遮罩語言模型的原理」
```

**Multi-Query 的解法：** 自動將原始查詢改寫成 N 種不同措辭，對每種都進行搜索，最後取**聯集（Union）**並去重。

```
原始查詢
    ↓ LLM 改寫
  [變體 1] → 搜索 → {doc_A, doc_B}
  [變體 2] → 搜索 → {doc_B, doc_C}   → 去重聯集 → {doc_A, doc_B, doc_C, doc_D}
  [變體 3] → 搜索 → {doc_C, doc_D}
  [原始]   → 搜索 → {doc_A, doc_D}
```

### 直覺解釋

就像你在 Google 找不到答案，換個關鍵字再搜一次——但 Multi-Query 讓 LLM 自動幫你想出所有可能的「換法」。

In [ ]:
def generate_query_variations(query: str, n: int = 3) -> list[str]:
    """讓 LLM 生成 N 種查詢的不同措辭"""
    prompt = f"""請將以下問題改寫成 {n} 種不同的措辭，保持相同的核心語意。
目的是用不同角度表達同一個問題，提升檢索系統的召回率。
每行輸出一個改寫，不要有編號或額外說明。

原始問題：{query}"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.8   # 較高溫度，讓改寫更多樣
    )
    lines = response.choices[0].message.content.strip().split("\n")
    # 過濾空行，確保輸出乾淨
    return [l.strip() for l in lines if l.strip()][:n]


def multi_query_retrieve(query: str, n_variations: int = 3, n_results_each: int = 3) -> dict:
    """Multi-Query 檢索：多種查詢取聯集"""
    # 生成查詢變體（包含原始查詢）
    variations = generate_query_variations(query, n=n_variations)
    all_queries = [query] + variations  # 原始查詢 + 變體

    seen_ids = set()
    all_results = []

    for q in all_queries:
        results = standard_retrieve(q, n_results=n_results_each)
        for r in results:
            if r["id"] not in seen_ids:  # 去重
                seen_ids.add(r["id"])
                all_results.append(r)

    return {
        "queries": all_queries,
        "results": all_results,
        "unique_doc_count": len(all_results)
    }


print("Multi-Query 函式已定義 ✅")

In [ ]:
# 示範：措辭敏感查詢的 Multi-Query 效果
phrasing_query = "BERT 的訓練方式"

print(f"查詢：{phrasing_query}")
print("=" * 60)

# 標準搜索（baseline）
standard_results = standard_retrieve(phrasing_query, n_results=3)
std_ids = [r["id"] for r in standard_results]
print(f"\n❶ 標準搜索找到 {len(standard_results)} 份文件：{std_ids}")
for r in standard_results:
    print(f"   [{r['id']}] 分數={r['score']}")

# Multi-Query 搜索
mq_result = multi_query_retrieve(phrasing_query, n_variations=3)

print(f"\n❷ Multi-Query 生成的查詢變體：")
for i, q in enumerate(mq_result["queries"]):
    label = "[原始]" if i == 0 else f"[變體 {i}]"
    print(f"   {label} {q}")

print(f"\n❸ Multi-Query 聯集結果（共 {mq_result['unique_doc_count']} 份，去重後）：")
mq_ids = [r["id"] for r in mq_result["results"]]
for r in mq_result["results"]:
    print(f"   [{r['id']}] 分數={r['score']} | {r['content'][:50]}...")

# 召回率提升分析
new_docs = set(mq_ids) - set(std_ids)
print(f"\n📊 召回率提升：標準={len(std_ids)} 份，Multi-Query={len(mq_ids)} 份")
if new_docs:
    print(f"   Multi-Query 額外找到：{new_docs}")
else:
    print("   兩者找到相同文件集合")

In [ ]:
# 展示 Multi-Query 的代價分析
n_variations = 3
n_all_queries = 1 + n_variations  # 原始 + 3 個變體

print("Multi-Query 的代價分析：")
print("-" * 40)
print(f"變體數量          : {n_variations}")
print(f"總查詢次數        : {n_all_queries} 次向量搜索")
print(f"額外 LLM 呼叫     : 1 次（生成變體）")
print(f"延遲增加（估計）  : +200~500ms（LLM）+ {n_variations}x 向量搜索")
print()
print("何時值得：")
print("  ✅ 召回率比延遲更重要的場景（如離線批次處理）")
print("  ✅ 知識庫中有多種術語表達相同概念")
print("  ❌ 即時對話系統（延遲敏感）")
print("  ❌ 查詢本身已非常精確")

### Multi-Query 重點總結

| 面向 | 說明 |
|------|------|
| **優勢** | 大幅提升召回率；對措辭不敏感；實作簡單 |
| **劣勢** | 搜索次數 = (1 + N) 倍；可能引入不相關文件（精確度下降） |
| **最適場景** | 召回率要求高、知識庫術語多樣 |
| **不適場景** | 延遲敏感系統、精確查詢 |
| **常見改進** | 配合 Reranking 過濾聯集中不相關的文件 |

---
## 策略三：Step-Back Prompting（後退提問）

### 核心概念

**問題所在：** 某些具體問題需要**背景知識**才能回答，但知識庫中的背景資訊可能不會直接和問題配對到。

例子：
- 具體問題：「GPT-4 在 2023 年 3 月的律師考試得了幾分？」
- 需要的背景：「大型語言模型在專業考試中的一般表現」

**Step-Back 的解法：** 先將具體問題「後退」一步，生成一個**更抽象、更通用**的問題，檢索這個大問題的背景知識，再結合原始具體問題的檢索結果。

```
具體問題：「GPT-4 在 2023 年 3 月律師考試得了幾分？」
    ↓ Step-Back
抽象問題：「大型語言模型在專業認證考試的表現如何？」
    ↓
同時檢索兩個問題 → 合併 context → 回答具體問題
```

### 直覺解釋

就像你問一位專家「今天 NVIDIA 股價為何上漲 5%？」，好的專家不會直接猜，而是先「後退」一步思考「半導體股的一般漲跌因素」，再結合具體新聞給出有深度的分析。

In [ ]:
def generate_step_back_query(query: str) -> str:
    """將具體查詢抽象化為更通用的 Step-Back 問題"""
    prompt = f"""你的任務是將一個具體問題改寫成更抽象、更通用的背景問題。
這個背景問題應該涵蓋原始問題的大類別，有助於提供必要的背景知識。

範例：
  具體問題：「Python 3.11 的 dict 的 get() 方法如何使用？」
  背景問題：「Python 字典的資料結構與操作方法」

具體問題：{query}
背景問題："""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3   # 低溫度，讓抽象化更穩定
    )
    return response.choices[0].message.content.strip()


def step_back_retrieve(query: str, n_results: int = 2) -> dict:
    """Step-Back 檢索：同時檢索原始問題和抽象問題，合併 context"""
    # 生成後退問題
    step_back_query = generate_step_back_query(query)

    # 分別檢索
    original_results = standard_retrieve(query, n_results=n_results)
    step_back_results = standard_retrieve(step_back_query, n_results=n_results)

    # 合併去重（保留兩份搜索的結果）
    seen_ids = set()
    combined = []
    for r in original_results + step_back_results:
        if r["id"] not in seen_ids:
            seen_ids.add(r["id"])
            combined.append(r)

    return {
        "original_query": query,
        "step_back_query": step_back_query,
        "original_results": original_results,
        "step_back_results": step_back_results,
        "combined_results": combined
    }


print("Step-Back 函式已定義 ✅")

In [ ]:
# 示範：需要背景知識的具體問題
specific_query = "GPT-4 在 2023 年 3 月律師考試得了幾分？"

print(f"具體問題：{specific_query}")
print("=" * 60)

# 標準搜索
standard_results = standard_retrieve(specific_query, n_results=3)
print("\n❶ 標準搜索結果：")
for r in standard_results:
    print(f"   [{r['id']}] 分數={r['score']} | {r['content'][:55]}...")

# Step-Back 搜索
sb_result = step_back_retrieve(specific_query, n_results=2)

print(f"\n❷ Step-Back 生成的抽象問題：")
print(f"   {sb_result['step_back_query']}")

print(f"\n❸ 抽象問題的搜索結果：")
for r in sb_result["step_back_results"]:
    print(f"   [{r['id']}] 分數={r['score']} | {r['content'][:55]}...")

print(f"\n❹ 合併後的 context（原始 + 抽象，共 {len(sb_result['combined_results'])} 份）：")
for r in sb_result["combined_results"]:
    print(f"   [{r['id']}] {r['content'][:55]}...")

In [ ]:
def answer_with_step_back(query: str) -> str:
    """使用 Step-Back 策略取得 context 並生成最終答案"""
    sb = step_back_retrieve(query, n_results=2)

    # 組合 context
    context_parts = []
    for r in sb["combined_results"]:
        context_parts.append(r["content"])
    context = "\n\n".join(context_parts)

    prompt = f"""請根據以下背景資料回答問題。

背景資料：
{context}

問題：{query}
請用繁體中文回答，若背景資料不足請誠實說明。"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1
    )
    return response.choices[0].message.content.strip()


# 生成最終回答
print("Step-Back 策略最終回答：")
print("-" * 50)
answer = answer_with_step_back(specific_query)
print(answer)

### Step-Back 重點總結

| 面向 | 說明 |
|------|------|
| **優勢** | 提供背景知識；對需要前提理解的問題特別有效 |
| **劣勢** | 抽象化可能偏離主題；增加額外 LLM 呼叫 |
| **最適場景** | 需要先理解原理才能回答的具體問題；學術或技術問答 |
| **不適場景** | 純事實查詢（直接搜索就夠）；文件庫已有充足背景資訊 |

> **論文來源：** Zheng et al., 2023, "Take a Step Back: Evoking Reasoning via Abstraction in Large Language Models"

---
## 策略四：Query Decomposition（查詢分解）

### 核心概念

**問題所在：** 複雜的多跳問題（Multi-hop questions）需要先回答子問題，才能回答主問題。單一查詢無法捕捉所有面向。

**Query Decomposition 的解法：** 將複雜問題拆解成**有序的子問題序列**，依序回答，並將前一個問題的答案作為下一個問題的輸入（Chain of Thought）。

```
複雜問題：「BERT 和 GPT-4 在多語言支援上各自有什麼優劣？」
    ↓ 分解
  子問題 1：「BERT 的多語言支援能力如何？」
      ↓ 搜索 + 回答 → answer1
  子問題 2：「GPT-4 的多語言支援能力如何？」  (可以使用 answer1 的上下文)
      ↓ 搜索 + 回答 → answer2
  子問題 3：「根據 answer1 和 answer2，兩者多語言能力的比較結論是？」
      ↓ 合成
  最終回答
```

### 直覺解釋

就像解數學大題，你不會直接從題目跳到答案，而是一步一步列出中間過程。查詢分解就是讓 RAG 系統也能「列出中間過程」。

In [ ]:
def decompose_query(query: str) -> list[str]:
    """將複雜問題分解為有序的子問題列表"""
    prompt = f"""請將以下複雜問題分解為 2-4 個有序的子問題，讓每個子問題更容易從知識庫中找到答案。
子問題應該是獨立的，並且按照邏輯順序排列（前一個問題的答案可以幫助回答後一個問題）。
每行輸出一個子問題，不要有編號或額外說明。

複雜問題：{query}"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )
    lines = response.choices[0].message.content.strip().split("\n")
    return [l.strip() for l in lines if l.strip()]


def answer_sub_question(sub_q: str, context: str, previous_answers: str = "") -> str:
    """回答單一子問題（可利用前面子問題的答案作為輔助）"""
    prev_context = f"\n\n前面子問題的答案：\n{previous_answers}" if previous_answers else ""

    prompt = f"""請根據以下資料回答子問題，回答要簡潔（2-4 句）。

知識庫資料：
{context}{prev_context}

子問題：{sub_q}"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1
    )
    return response.choices[0].message.content.strip()


def sequential_retrieve_and_answer(complex_query: str) -> dict:
    """Sequential Decomposition：依序回答子問題，串聯中間答案"""
    # 分解問題
    sub_questions = decompose_query(complex_query)

    results = []
    accumulated_answers = ""  # 累積前面子問題的答案

    for i, sub_q in enumerate(sub_questions):
        # 為每個子問題搜索 context
        retrieved = standard_retrieve(sub_q, n_results=2)
        context = "\n\n".join([r["content"] for r in retrieved])

        # 回答子問題（帶上前面的答案）
        sub_answer = answer_sub_question(sub_q, context, accumulated_answers)

        results.append({
            "sub_question": sub_q,
            "retrieved_ids": [r["id"] for r in retrieved],
            "answer": sub_answer
        })

        # 累積答案供後續子問題使用
        accumulated_answers += f"Q{i+1}: {sub_q}\nA{i+1}: {sub_answer}\n\n"

    # 最終合成答案
    synthesis_prompt = f"""請根據以下各子問題的答案，對原始複雜問題給出完整的綜合回答。

{accumulated_answers}

原始問題：{complex_query}
請用繁體中文給出完整、條理清晰的最終回答。"""

    final_response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": synthesis_prompt}],
        temperature=0.1
    )
    final_answer = final_response.choices[0].message.content.strip()

    return {
        "complex_query": complex_query,
        "sub_question_results": results,
        "final_answer": final_answer
    }


print("Query Decomposition 函式已定義 ✅")

In [ ]:
# 示範：複雜多跳問題的分解
complex_query = "BERT 和 GPT-4 在多語言支援上各自有什麼優劣？"

print(f"複雜問題：{complex_query}")
print("=" * 60)

# 先看分解結果
sub_qs = decompose_query(complex_query)
print("\n❶ 分解後的子問題：")
for i, q in enumerate(sub_qs, 1):
    print(f"   子問題 {i}：{q}")

# 執行完整流程
print("\n❷ 依序回答子問題...")
result = sequential_retrieve_and_answer(complex_query)

print()
for i, sr in enumerate(result["sub_question_results"], 1):
    print(f"   子問題 {i}：{sr['sub_question']}")
    print(f"   檢索文件：{sr['retrieved_ids']}")
    print(f"   子答案：{sr['answer'][:100]}...")
    print()

print("❸ 最終合成回答：")
print("-" * 50)
print(result["final_answer"])

In [ ]:
# 對比：標準 RAG vs 分解法
def simple_rag_answer(query: str) -> str:
    """最簡單的標準 RAG（直接搜索 + 生成）"""
    docs = standard_retrieve(query, n_results=3)
    context = "\n\n".join([r["content"] for r in docs])
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "根據提供的資料回答問題，用繁體中文。"},
            {"role": "user", "content": f"資料：\n{context}\n\n問題：{query}"}
        ],
        temperature=0.1
    )
    return response.choices[0].message.content.strip()


print("📊 對比：標準 RAG vs 查詢分解法")
print("=" * 60)
print(f"問題：{complex_query}")
print()

print("❌ 標準 RAG 回答：")
std_ans = simple_rag_answer(complex_query)
print(std_ans[:300], "..." if len(std_ans) > 300 else "")

print()
print("✅ 查詢分解法回答（已在上方顯示）")
print("觀察差異：分解法能分別取得 BERT 和 GPT-4 的專屬文件，合成更完整")

### Query Decomposition 重點總結

| 面向 | 說明 |
|------|------|
| **優勢** | 處理複雜多跳問題；中間結果可供解釋；Chain of Thought 提升推理品質 |
| **劣勢** | 最高代價（N 次 LLM + N 次搜索）；錯誤的子問題會傳播誤差 |
| **最適場景** | 比較類問題、需要多步推理、複雜技術問答 |
| **不適場景** | 簡單查詢、延遲敏感系統 |
| **進階版本** | Least-to-Most Prompting；Tree of Thought |

---
## 5. 四大策略全面比較

### 情境適用表

| 查詢類型 | 範例 | 推薦策略 | 備選 |
|----------|------|----------|------|
| **模糊/開放問題** | 「解釋量子電腦」 | HyDE | Multi-Query |
| **措辭敏感** | 「Bidirectional Encoder 訓練」 | Multi-Query | HyDE |
| **需要背景知識** | 「GPT-4 律師考試幾分？」 | Step-Back | — |
| **多跳比較問題** | 「BERT vs GPT-4 多語言能力」 | Decomposition | Multi-Query |
| **精確事實查詢** | 「HNSW 是什麼縮寫？」 | 標準搜索 | — |

### 策略決策樹

```
查詢是否很模糊/開放？
  是 → 用 HyDE
  否 ↓

查詢是否包含多個子問題或需要比較？
  是 → 用 Query Decomposition
  否 ↓

查詢是否需要背景知識才能理解？
  是 → 用 Step-Back Prompting
  否 ↓

是否高度關注召回率（寧可找多、不能找少）？
  是 → 用 Multi-Query
  否 → 用標準向量搜索即可
```

In [ ]:
# 在同一個查詢上執行所有四種策略並比較
comparison_query = "BERT 的訓練方法"

print(f"比較查詢：{comparison_query}")
print("=" * 60)

# 收集各策略的召回文件 ID
results_summary = {}

# 1. 標準搜索
std = standard_retrieve(comparison_query, n_results=3)
results_summary["標準搜索"] = [r["id"] for r in std]
print(f"\n1. 標準搜索      : {results_summary['標準搜索']}")

# 2. HyDE
hyde = hyde_retrieve(comparison_query, n_results=3)
results_summary["HyDE"] = [r["id"] for r in hyde["results"]]
print(f"2. HyDE          : {results_summary['HyDE']}")

# 3. Multi-Query
mq = multi_query_retrieve(comparison_query, n_variations=3, n_results_each=2)
results_summary["Multi-Query"] = [r["id"] for r in mq["results"]]
print(f"3. Multi-Query   : {results_summary['Multi-Query']}")

# 4. Step-Back
sb = step_back_retrieve(comparison_query, n_results=2)
results_summary["Step-Back"] = [r["id"] for r in sb["combined_results"]]
print(f"4. Step-Back     : {results_summary['Step-Back']}")

print("\n唯一文件召回數量比較：")
for strategy, ids in results_summary.items():
    print(f"  {strategy:12s}: {len(set(ids))} 份 → {sorted(set(ids))}")

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 設定支援中文的字體（macOS 適用）
chinese_fonts = [f.name for f in fm.fontManager.ttflist
                 if any(kw in f.name for kw in ["Heiti", "PingFang", "Noto", "CJK", "Source Han", "WenQuanYi"])]
if chinese_fonts:
    plt.rcParams["font.family"] = chinese_fonts[0]
else:
    plt.rcParams["font.family"] = "DejaVu Sans"  # fallback

plt.rcParams["axes.unicode_minus"] = False

# 比較不同策略的召回文件數量（橫條圖）
strategies = list(results_summary.keys())
recall_counts = [len(set(ids)) for ids in results_summary.values()]
colors = ["#5B9BD5", "#ED7D31", "#A9D18E", "#FFC000"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 左圖：召回文件數量
ax1 = axes[0]
bars = ax1.bar(strategies, recall_counts, color=colors, width=0.5, edgecolor="white", linewidth=1.2)
ax1.set_title("各策略召回文件數量比較", fontsize=14, pad=12)
ax1.set_ylabel("唯一文件數量", fontsize=11)
ax1.set_ylim(0, max(recall_counts) + 2)
ax1.set_xticks(range(len(strategies)))
ax1.set_xticklabels(strategies, fontsize=10)
for bar, count in zip(bars, recall_counts):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
             str(count), ha="center", va="bottom", fontsize=12, fontweight="bold")
ax1.grid(axis="y", alpha=0.3)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)

# 右圖：代價（LLM 呼叫次數）vs 召回提升
ax2 = axes[1]
llm_calls   = [0, 1, 1, 1]   # 各策略額外 LLM 呼叫次數
# 計算相對於標準搜索的召回提升率
base_recall = recall_counts[0] if recall_counts[0] > 0 else 1
recall_gain = [round((c / base_recall - 1) * 100, 1) for c in recall_counts]

scatter_colors = colors
for i, (x, y, name) in enumerate(zip(llm_calls, recall_gain, strategies)):
    ax2.scatter(x, y, s=200, color=scatter_colors[i], zorder=5, edgecolors="white", linewidths=1.5)
    offset_y = 3 if i != 2 else -8
    ax2.annotate(name, (x, y), textcoords="offset points",
                 xytext=(10, offset_y), fontsize=10)

ax2.set_title("代價 vs 召回提升率", fontsize=14, pad=12)
ax2.set_xlabel("額外 LLM 呼叫次數", fontsize=11)
ax2.set_ylabel("召回提升率（%）", fontsize=11)
ax2.axhline(0, color="gray", linestyle="--", alpha=0.5)
ax2.set_xlim(-0.3, 1.5)
ax2.grid(alpha=0.3)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)

plt.suptitle(f'Query 轉換策略比較（查詢："{comparison_query}"）', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("/tmp/query_transform_comparison.png", dpi=120, bbox_inches="tight")
plt.show()
print("圖表已生成")

In [ ]:
# 代價矩陣總覽
print("四大策略代價矩陣")
print("=" * 70)
header = f"{'策略':<14} {'額外LLM呼叫':>10} {'向量搜索倍數':>12} {'延遲增加(估計)':>16} {'最適場景'}"
print(header)
print("-" * 70)

rows = [
    ("標準搜索",     "0 次",  "1x",   "基準",           "精確查詢"),
    ("HyDE",         "+1 次", "1x",   "+200~500ms",     "模糊/開放問題"),
    ("Multi-Query",  "+1 次", "(1+N)x","+N×搜索延遲",  "措辭不確定、高召回率"),
    ("Step-Back",    "+1 次", "2x",   "+200~500ms",     "需要背景知識"),
    ("Query Decomp.","+N 次", "Nx",   "+N×(LLM+搜索)",  "複雜多跳問題"),
]

for r in rows:
    print(f"{r[0]:<14} {r[1]:>10} {r[2]:>12} {r[3]:>16}   {r[4]}")

---
## 6. 面試關鍵問答（FDE Interview Q&A）

以下是 Google FDE 面試中常見的 Query Transformation 相關問題，以及高分回答範例。

---

**面試官：** 使用者輸入的問題品質很差、很模糊，怎麼辦？

**好答案：** 「這是 RAG 系統的關鍵挑戰。我會根據問題類型選擇不同策略：如果問題很模糊，用 **HyDE**——讓 LLM 先生成一個假設性答案，再用答案向量搜索，因為答案向量和文件向量天然更相似。如果問題用語不確定，用 **Multi-Query**——自動生成多種措辭的查詢，取聯集，從根本上消除對特定措辭的依賴。這比要求使用者重新表達問題更好的使用者體驗。」

---

**面試官：** HyDE 的原理是什麼？為什麼它有效？

**好答案：** 「HyDE（Hypothetical Document Embeddings）利用了一個觀察：查詢和文件在嵌入空間中天然有 **asymmetry（不對稱性）**——查詢短、口語化，文件長、術語豐富，兩者向量風格不同。HyDE 的解法是讓 LLM 先生成一個假設性答案，這個假設文件的風格和真實文件類似，嵌入後的向量距離更近。即使假設文件並不 100% 正確，只要它的術語和概念方向對，就能找到相關文件。」

---

**面試官：** Multi-Query 和 HyDE 有什麼差別，什麼時候用哪個？

**好答案：** 「兩者解決不同的問題。**Multi-Query** 解決的是**措辭不確定性**——同一個問題不同表達方式，確保不因為措辭不同而漏掉相關文件，召回率（Recall）導向。**HyDE** 解決的是**查詢-文件空間不匹配**——用答案風格的向量去搜索，不管使用者怎麼問，都能提升和文件的向量相似度，適合模糊問題。如果問題模糊且需要高召回率，可以同時用兩者，但代價更高。」

---

**面試官：** 如果 RAG 系統需要回答複雜的比較問題（如「A 和 B 有什麼不同？」），你會怎麼設計？

**好答案：** 「比較類問題是 RAG 的難點，因為需要同時了解 A 和 B 的資訊，但單一查詢通常只能偏向其中一邊。我會使用 **Query Decomposition（查詢分解）**：先把問題拆成「A 是什麼？」和「B 是什麼？」，分別搜索、分別回答，然後讓 LLM 做最終的比較合成。這樣能確保 A 和 B 的資訊都被充分檢索，最終的比較答案更完整。如果知識庫有專屬的比較文件，也可以額外加入比較類的查詢維度。」

---

**面試官：** 這些 Query 轉換技術有什麼共同的代價？在生產系統中如何取捨？

**好答案：** 「這些技術的共同代價是**延遲（Latency）和成本（Cost）**——都需要額外的 LLM 呼叫，HyDE/Step-Back 各加 ~200-500ms，Multi-Query/Decomposition 加更多。在生產系統中，我的取捨原則是：1) **查詢分類**：先判斷查詢類型，只對需要的查詢用對應策略，不一律套用。2) **非同步前處理**：對可預測的查詢預先做轉換和快取。3) **成本監控**：監控每種策略的 p95 延遲和 LLM 呼叫成本，設定上限。4) **降級策略**：超過延遲預算時，降回標準搜索，保證服務可用性。」

---

---
## 7. 總結與決策指南

### 本 Notebook 學習成果

1. **HyDE**：嵌入假設答案向量而非查詢向量，橋接 query-document asymmetry
2. **Multi-Query**：自動改寫 N 種措辭，取聯集，消除措辭敏感性
3. **Step-Back**：先問大問題取背景知識，再結合小問題搜索
4. **Query Decomposition**：拆解多跳問題，串聯中間答案，提升複雜推理品質

### 查詢類型 → 推薦策略速查表

```
查詢類型                   推薦策略              次選策略
──────────────────────────────────────────────────────────
模糊/開放問題               HyDE               Multi-Query
措辭不確定（同義詞多）      Multi-Query         HyDE
需要背景/前提知識           Step-Back           ─
多跳/比較/推理問題          Query Decomposition Multi-Query
精確事實查詢                標準搜索（直接用）  ─
延遲極敏感                  標準搜索            ─
```

### 下一步

Query Transformation 通常和其他技術組合使用：
- **NB03 的 Reranking**：Multi-Query 召回的聯集用 Reranker 過濾，精確度更高
- **NB04 的 RAG 評估框架**：用 RAGAS 評估不同策略的 context recall 和 answer relevancy
- **下一步**：Adaptive RAG — 根據查詢特徵自動選擇最佳策略的元系統

In [ ]:
# 最終綜合示範：自動選擇策略
def auto_select_strategy(query: str) -> str:
    """根據查詢特徵自動選擇最佳 Query 轉換策略（示範用）"""
    prompt = f"""分析以下查詢，從四個策略中選一個最適合的，只輸出策略名稱：
  - standard（精確事實查詢）
  - hyde（模糊/開放問題）  
  - multi_query（可能有多種措辭的問題）
  - step_back（需要背景知識的問題）
  - decomposition（多跳/比較/複雜推理問題）

查詢：{query}
策略（只輸出一個名稱）："""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1,
        max_tokens=20
    )
    return response.choices[0].message.content.strip().lower()


# 測試自動選擇
test_queries = [
    "HNSW 索引是什麼？",
    "解釋一下 Transformer 的運作",
    "GPT-4 在 SAT 數學考試的成績",
    "BERT 和 GPT-4 的多語言能力有何差異？",
]

print("自動策略選擇示範：")
print("-" * 55)
for q in test_queries:
    strategy = auto_select_strategy(q)
    print(f"  查詢：{q[:35]:<35} → {strategy}")